In [1]:
%reload_ext autoreload
from abc import ABC, abstractmethod
import torch
from torch import nn


class AlignmentStrategy(ABC):
    """
    The parent class for different type of alignment computations.
    """
    def __init__(self):
        self.relative = True

    @abstractmethod
    def compute(self, input, weight, ouput):
        """
        """
        pass

class CovarianceAlignment(AlignmentStrategy):
    def compute(self, input, weight, output):
        cc = torch.cov(input.T)
        rq = torch.sum(torch.matmul(weight, cc) * weight, axis=1) / torch.sum(weight * weight, axis=1)
        
        if self.relative:
            # proportion of variance explained by a projection of the input onto each weight
            return rq / torch.trace(cc)
        
        return rq


class AligNet(nn.Module):
    """
    This is a wrapper class for alignment network decoupling the main model. Will play the main role of managing the alignment computations for different models.
    The main model will be add as an attribute to this class.
    """
    def __init__(self,
                 model: nn.Module,
                 alignment_layer_names=None,
                 alignment_strategy: AlignmentStrategy = CovarianceAlignment,
                 ):
        super().__init__()

        self.model = model
        self.alignment_strategy = alignment_strategy # .compute() to be added into the hook function

        self.alignment_layer_names = alignment_layer_names
        self.alignment_values = {}
        self.alignment_activations = {}
        self.hooks = {}
        

    def __getattr__(self, name):
        # Try getting the attribute from nn.Module or AlignmentNetwork first
        try:
            return super().__getattr__(name)
        except AttributeError:
        # Redirect attribute access to the model if the attribute is not found in the wrapper class
            return getattr(self.model, name)

    def forward(self, x):
        return self.model(x)
    
    def preprocessing(self, module, input, output):

        # TODO: do some pre-process on the input
        if type(module) == torch.nn.modules.conv.Conv2d:
            layer_prms = dict(stride=module.stride, padding=module.padding, dilation=module.dilation) 
            unfolded_input = torch.nn.functional.unfold(input, module.kernel_size, **layer_prms)
            input = unfolded_input.transpose(1, 2).contiguous().view(-1, unfolded_input.size(1))

        weight = module.weight.data.clone()
        weight = weight.flatten(start_dim=1)

        return input, weight, output

    def get_alignment_values(self):
        return self.alignment_values
    
    @torch.no_grad()
    def get_alignment_weights(self, idx=None, flatten=False):
        """
        convenience method for retrieving registered weights for alignment measurements throughout the network

        if flatten=True, will flatten weights so they have shape (nodes/channels, numel_per_weight)
        """
        # go through each layer and retrieve weight as desired
        weights = []
        for name, layer in self.model.named_modules():
            if not (self.alignment_layer_names is None or name in self.alignment_layer_names): continue
            # get weight data for this layer
            weight = layer.weight.data.clone()

            # if requesting flat weights, flatten them
            if flatten:
                weight = weight.flatten(start_dim=1)

            # add weights to list
            weights.append(weight)

        # return
        if idx is None:
            return weights
        return weights[idx]

    def setup_forward_hooks(self):
        
        def getActivation(name):
            def activation_hook(module, input, output):
                self.alignment_activations[name] = output.detach()
            return activation_hook
        
        def getAlignment(name):
            def alignment_hook(module, input, output):
                passed_input = input[0].detach() if self.alignment_layer_names is None else self.alignment_activations[self.alignment_layer_names[name]]
                passed_input, weight, output = self.preprocessing(module, passed_input, output.detach())
                self.alignment_values[name] = self.alignment_strategy.compute(passed_input, weight, output)
            return alignment_hook

        self.hooks = {}
        for name, module in self.model.named_modules():
            if self.alignment_layer_names is None:
                if not hasattr(module, 'weight'): continue
                print(f"setup hook to compute RQ of layer {name}")
                self.hooks[name] = module.register_forward_hook(getAlignment(name))
            else:
                if name in self.alignment_layer_names.values():
                    print(f"setup hook to get output of layer {name}")
                    self.hooks[name] = module.register_forward_hook(getActivation(name))
                if name in self.alignment_layer_names.keys():
                    print(f"setup hook to compute RQ of layer {name}")
                    self.hooks[name] = module.register_forward_hook(getAlignment(name))
                
    
    def remove_forward_hooks(self):
        for _, hook in self.hooks.items():
            hook.remove()

In [2]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('using device: ', DEVICE)

using device:  cuda


In [3]:
%autoreload 2
from pathlib import Path

from torchvision.models import alexnet

from alignment.datasets import get_dataset

base_model = alexnet(weights='DEFAULT')

transform_parameters = {
    "center_crop": 224,
    "flatten": False,
    "resize": (256, 256),
}

loader_parameters = dict(
    batch_size = 1024,
)

data_path = "/n/holylfs06/LABS/kempner_shared/Lab/data/imagenet_1k/"
dataset_parameters = dict(
    root=Path(data_path),
)

dataset = get_dataset("ImageNet", build=True, transform_parameters=transform_parameters, loader_parameters=loader_parameters, dataset_parameters=dataset_parameters, device=DEVICE)

In [4]:
# extracting the layers name and architecture
def print_module_hierarchy(module, indent=0, prefix=''):
    for name, submodule in module.named_children():
        name = prefix + '.' + name if prefix else name
        print("  " * indent + f"{name}: {submodule.__class__.__name__}")
        print_module_hierarchy(submodule, indent + 1, name)

print_module_hierarchy(base_model)

features: Sequential
  features.0: Conv2d
  features.1: ReLU
  features.2: MaxPool2d
  features.3: Conv2d
  features.4: ReLU
  features.5: MaxPool2d
  features.6: Conv2d
  features.7: ReLU
  features.8: Conv2d
  features.9: ReLU
  features.10: Conv2d
  features.11: ReLU
  features.12: MaxPool2d
avgpool: AdaptiveAvgPool2d
classifier: Sequential
  classifier.0: Dropout
  classifier.1: Linear
  classifier.2: ReLU
  classifier.3: Dropout
  classifier.4: Linear
  classifier.5: ReLU
  classifier.6: Linear


In [5]:
from torchinfo import summary

mm = summary(base_model)
mm

Layer (type:depth-idx)                   Param #
AlexNet                                  --
├─Sequential: 1-1                        --
│    └─Conv2d: 2-1                       23,296
│    └─ReLU: 2-2                         --
│    └─MaxPool2d: 2-3                    --
│    └─Conv2d: 2-4                       307,392
│    └─ReLU: 2-5                         --
│    └─MaxPool2d: 2-6                    --
│    └─Conv2d: 2-7                       663,936
│    └─ReLU: 2-8                         --
│    └─Conv2d: 2-9                       884,992
│    └─ReLU: 2-10                        --
│    └─Conv2d: 2-11                      590,080
│    └─ReLU: 2-12                        --
│    └─MaxPool2d: 2-13                   --
├─AdaptiveAvgPool2d: 1-2                 --
├─Sequential: 1-3                        --
│    └─Dropout: 2-14                     --
│    └─Linear: 2-15                      37,752,832
│    └─ReLU: 2-16                        --
│    └─Dropout: 2-17                   

In [6]:
alignment_strategy = CovarianceAlignment()

alignment_layer_names={"features.6": "features.5"}

model = AligNet(model=base_model,
                alignment_layer_names=alignment_layer_names,
                alignment_strategy=alignment_strategy,
                ).to(DEVICE)

In [19]:
batch_loop = dataset.train_loader

model.eval()
model.setup_forward_hooks()
rq_val=[]
step = 0
end = 2
for idx, batch in enumerate(batch_loop):
    images, labels = dataset.unwrap_batch(batch)
    with torch.no_grad():
        model(images)
    alignment_values = model.get_alignment_values()
    rq_val.append(list(alignment_values.values())[0])
    step += 1
    if step == end: break
model.remove_forward_hooks()    

setup hook to get output of layer features.5
setup hook to compute RQ of layer features.6


In [26]:
avg_rq = sum(rq_val) / len(rq_val)